# `specsynth`: Cloud-Native Stellar Population Synthesis Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SankarGaneshb/specsynth/blob/main/notebooks/quickstart.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)

This notebook demonstrates the end-to-end forward modeling of **5,000 simulated galaxies** into **JWST NIRCam broadband photometry** at cosmological redshift $z = 2.0$ with **Madau (1995) IGM attenuation**, using `specsynth`.

## 1. Installation
Install `specsynth` directly from GitHub alongside plotting utilities.

In [ ]:
# Install specsynth directly in Kaggle / Google Colab
!pip install -q git+https://github.com/SankarGaneshb/specsynth.git matplotlib
import shutil, subprocess
if shutil.which("nvidia-smi"):
    try:
        import cupy
        print("NVIDIA GPU + CuPy already configured cleanly!")
    except ImportError:
        print("NVIDIA GPU detected! Installing matching CuPy package...")
        subprocess.run(["pip", "install", "-q", "cupy-cuda12x"], check=False)
print("specsynth successfully installed!")

## 2. Compile Simple Stellar Population (SSP) Library
We compile a stellar population grid (40 age bins $\times$ 4 metallicity bins $\times$ 2,000 wavelength bins) into chunked Zarr storage.

In [ ]:
import os
import subprocess

RUN_DIR = "./run_demo"
os.makedirs(RUN_DIR, exist_ok=True)

# Step 1: Compile SSP Library (Toy adapter or BC03/E-MILES/FSPS)
!specsynth --base {RUN_DIR} make-ssp --adapter toy --n-age 40 --n-met 4 --n-wave 2000

## 3. Generate or Ingest Galaxy Star Formation Histories
Generate 5,000 galaxies with realistic delayed-$\tau$ SFH, stellar masses, metallicities, and Charlot & Fall (2000) two-component dust screens.

In [ ]:
# Step 2: Generate 5,000 simulated galaxies
!specsynth --base {RUN_DIR} make-sfh --n-gal 5000 --chunk 2500 --seed 42

## 4. Distributed Synthesis with JWST NIRCam Filters & Madau IGM
Synthesize broadband photometry for JWST NIRCam (`jwst_nircam`) at $z = 2.0$ with IGM attenuation enabled.

In [ ]:
# Step 3: Synthesize photometry in-flight without saving multi-gigabyte spectra
!specsynth --base {RUN_DIR} synth --filters jwst_nircam --redshift 2.0 --igm

## 5. Visualizing the Synthetic JWST Color-Magnitude Diagram (CMD)
Let's load the generated Zarr array and inspect the resulting JWST NIRCam color-magnitude distribution ($m_{\text{F200W}} - m_{\text{F444W}}$ vs. $m_{\text{F200W}}$).

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt

# Open synthesized photometry
phot_store = f"{RUN_DIR}/spectra.zarr"
try:
    root = zarr.open_group(phot_store, mode='r')
    phot = root["photometry"][:]
    filters = root.attrs.get("filters", ["F090W", "F115W", "F150W", "F200W", "F277W", "F356W", "F410M", "F444W"])
except Exception:
    import dask.array as da
    phot = da.from_zarr(phot_store, component="photometry").compute()
    filters = ["F090W", "F115W", "F150W", "F200W", "F277W", "F356W", "F410M", "F444W"]

print(f"Photometry matrix loaded: {phot.shape} (Galaxies x Filters)")
print("Filter order:", filters)

# Convert flux densities to AB magnitudes
# m_AB = -2.5 * log10(flux) - 48.60 (or relative AB offsets)
f200w_idx = filters.index("jwst_f200w") if "jwst_f200w" in filters else 3
f444w_idx = filters.index("jwst_f444w") if "jwst_f444w" in filters else -1

flux_f200w = np.clip(phot[:, f200w_idx], 1e-35, None)
flux_f444w = np.clip(phot[:, f444w_idx], 1e-35, None)

mag_f200w = -2.5 * np.log10(flux_f200w) - 48.60
mag_f444w = -2.5 * np.log10(flux_f444w) - 48.60
color_f200_f444 = mag_f200w - mag_f444w

# Plot Publication-Quality Color-Magnitude Diagram
plt.style.use('default')
fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

scatter = ax.scatter(
    color_f200_f444, mag_f200w,
    c=color_f200_f444, cmap='viridis',
    s=15, alpha=0.6, edgecolors='none'
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("JWST NIRCam F200W - F444W Color [AB mag]", fontsize=11)

ax.set_xlabel("Color: F200W - F444W [AB mag]", fontsize=12, fontweight='bold')
ax.set_ylabel("Apparent Magnitude: F200W [AB mag]", fontsize=12, fontweight='bold')
ax.invert_yaxis()  # Brighter objects at the top
ax.set_title("Synthetic JWST NIRCam Color-Magnitude Diagram ($z = 2.0$, Madau IGM)", fontsize=13, pad=12)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 6. Reproducible Local Hardware Benchmark
Run `specsynth benchmark` to profile your local hardware throughput and GFLOPS rate.

In [ ]:
!specsynth benchmark --n-gal 50000 --n-runs 5 --gpu --output gpu_manifest.json
import json
with open("gpu_manifest.json") as f:
    print("\n=== GPU BENCHMARK MANIFEST (JSON) ===")
    print(f.read())